# Feature extraction 01 — Eye tracking

**Input:** the merged 25 Hz file (`physio_eye_survey.csv`, ~2.5 GB, not included in the repository).
**Output:** `data/eye_features_by_block.csv` — one row per Commander × reliability block: 21 eye-tracking features plus the Commander's post-block
trust ratings of each UAV (MDMT), used as targets in later notebooks.

Only the Commanders wore eye-tracking glasses (Pupil Labs Neon), so every row describes a Commander.
The model notebooks in `models/` start from the output file and do not need the raw data.

Steps:
1. Keep the three RSTA task blocks (High-High, Mixed, Low-Low).
2. Rebuild fixation and saccade **events** from the 25 Hz grid (each event spans several rows).
3. Apply fixed event filters taken from the literature (fixation 80–4000 ms; saccade ≤ 120 ms).
4. Summarise each block into pupil, fixation, area-of-interest (AOI), saccade and gaze-transition features.

In [1]:
import os
from pathlib import Path
# Run from anywhere inside the repository: paths below are relative to the repository root
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "feature_extraction").is_dir())
os.chdir(REPO)

# Settings
import os
import numpy as np
import pandas as pd

RAW_PATH = "/Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/physio_eye_survey.csv"
OUT_PATH = "data/eye_features_by_block.csv"

# Event filters: fixed thresholds adopted from the literature (not tuned on this dataset)
FIX_MIN_MS, FIX_MAX_MS = 80, 4000      # Jia & Du (2026), after Dybvik et al. (2023)
SAC_MAX_DUR_MS = 120                   # Ries et al. (2026), after Enders et al. (2021); Long et al. (2024)
# No peak-velocity filter: the literature thresholds are in deg/s, but Pupil Cloud exports peak velocity
# only in scene-camera px/s, and there is no validated px/s -> deg/s conversion for this export.

In [2]:
# Load only the columns needed, keep the three task blocks
cols = ["ID", "Condition", "Reliability", "unix_ns",
        "Fix_ID", "Fix_Duration_ms", "AOI_North", "AOI_South",
        "Sac_ID", "Sac_Duration_ms", "Sac_Amplitude_deg", "Sac_PeakVel_px_s",
        "Pupil_clean", "Pupil_blink_frac", "MDMT_North_C", "MDMT_South_C"]
raw = pd.read_csv(RAW_PATH, usecols=cols, engine="pyarrow")
task = raw[raw["Reliability"].isin(["HH", "M", "LL"])].copy()
print(f"{len(task):,} task rows | {task['ID'].nunique()} Commanders")

2,385,785 task rows | 51 Commanders


## Feature functions

**Gaze transitions** (Cui et al., 2024; Krejtz et al., 2015). Fixations are labelled North (N), South (S) or Other (O)
and put in time order. From consecutive pairs:

- `p_N_to_S`, `p_S_to_N` — probability that the next fixation is on the other UAV
- `NS_switch_rate` — direct North↔South switches per minute
- `SGE` — stationary gaze entropy (how evenly fixations are spread over N/S/O), scaled 0–1
- `GTE` — gaze transition entropy (how unpredictable the next AOI is), scaled 0–1

Transitions include repeat fixations on the same AOI (e.g., N→N).

**Saccades** are taken from the Pupil Cloud export, where a saccade is the gap between two consecutive fixations
(Pupil Labs documentation). Amplitude is exported in degrees; peak velocity only in scene-camera pixels per second,
so `sac_peakvel_px_mean` stays in px/s (not converted).

In [3]:
AOIS = ["N", "S", "O"]   # North, South, Other

def transition_features(aoi_seq):
    # Gaze transition measures from a time-ordered sequence of fixation AOIs
    idx = pd.Series(aoi_seq).map({a: k for k, a in enumerate(AOIS)}).to_numpy()
    if len(idx) < 2:
        return {k: np.nan for k in ["SGE", "GTE", "p_N_to_S", "p_S_to_N", "NS_switches"]}
    M = np.zeros((3, 3))                                          # M[i, j] = fixation on i followed by j
    np.add.at(M, (idx[:-1], idx[1:]), 1)
    p_i = np.bincount(idx, minlength=3) / len(idx)                # share of fixations on each AOI
    row = M.sum(axis=1, keepdims=True)
    P = np.divide(M, row, out=np.zeros_like(M), where=row > 0)    # P[i, j] = P(next = j | current = i)
    h = lambda p: -(p[p > 0] * np.log2(p[p > 0])).sum()           # Shannon entropy (bits)
    return {"SGE": h(p_i) / np.log2(3),
            "GTE": sum(p_i[i] * h(P[i]) for i in range(3)) / np.log2(3),
            "p_N_to_S": P[0, 1], "p_S_to_N": P[1, 0],
            "NS_switches": M[0, 1] + M[1, 0]}


def block_features(task):
    # One row per Commander x block, North and South features as separate columns
    keys = ["ID", "Reliability"]

    # pupil and block length
    B = task.groupby(keys).agg(
        Condition=("Condition", "first"), t_start=("unix_ns", "min"), t_end=("unix_ns", "max"),
        pupil_mean=("Pupil_clean", "mean"), pupil_sd=("Pupil_clean", "std"),
        blink_frac=("Pupil_blink_frac", "mean"),
        MDMT_North_C=("MDMT_North_C", "first"), MDMT_South_C=("MDMT_South_C", "first")).reset_index()
    B["minutes"] = (B["t_end"] - B["t_start"]) / 6e10 + 0.04 / 60
    B["pupil_mean_wp"] = B["pupil_mean"] - B["ID"].map(task.groupby("ID")["Pupil_clean"].mean())

    # fixations: one row per event, duration filter, time order, AOI label
    fix = (task.dropna(subset=["Fix_ID"]).groupby(keys + ["Fix_ID"])
               .agg(t0=("unix_ns", "min"), dur=("Fix_Duration_ms", "first"),
                    north=("AOI_North", "max"), south=("AOI_South", "max"))
               .reset_index())
    assert not ((fix["north"] == 1) & (fix["south"] == 1)).any(), "a fixation touches both UAV AOIs"
    fix = fix[fix["dur"].between(FIX_MIN_MS, FIX_MAX_MS)].sort_values(keys + ["t0"])
    fix["aoi"] = np.select([fix["north"] == 1, fix["south"] == 1], ["N", "S"], default="O")
    parts = [fix.groupby(keys).agg(fix_n=("dur", "size"), fix_dur_mean=("dur", "mean"),
                                   fix_total_ms=("dur", "sum")).reset_index()]
    for uav, c in [("North", "N"), ("South", "S")]:
        parts.append(fix[fix["aoi"] == c].groupby(keys)
                     .agg(**{f"{uav}_fix_n": ("dur", "size"), f"{uav}_dwell_ms": ("dur", "sum"),
                             f"{uav}_fix_dur_mean": ("dur", "mean")}).reset_index())
    parts.append(fix.groupby(keys)["aoi"].apply(lambda s: pd.Series(transition_features(s.to_numpy())))
                    .unstack().reset_index())

    # saccades: one row per event; duration and peak-velocity filters
    sac = (task.dropna(subset=["Sac_ID"]).groupby(keys + ["Sac_ID"])
               .agg(dur=("Sac_Duration_ms", "first"), amp=("Sac_Amplitude_deg", "first"),
                    pv_px=("Sac_PeakVel_px_s", "first"))
               .reset_index())
    sac = sac[sac["dur"] <= SAC_MAX_DUR_MS]                               # duration filter only
    parts.append(sac.groupby(keys).agg(sac_n=("dur", "size"), sac_amp_mean=("amp", "mean"),
                                       sac_peakvel_px_mean=("pv_px", "mean"), sac_dur_mean=("dur", "mean"))
                    .reset_index())

    for p in parts:
        B = B.merge(p, on=keys, how="left")

    # counts -> rates and proportions
    has_fix = B["fix_n"].notna()
    for uav in ["North", "South"]:
        B[f"{uav}_fix_n"] = B[f"{uav}_fix_n"].fillna(0).where(has_fix)
        B[f"{uav}_dwell_ms"] = B[f"{uav}_dwell_ms"].fillna(0).where(has_fix)
        B[f"{uav}_dwell_frac"] = B[f"{uav}_dwell_ms"] / B["fix_total_ms"]
        B[f"{uav}_fix_rate"] = B[f"{uav}_fix_n"] / B["minutes"]
    B["fix_rate"] = B["fix_n"] / B["minutes"]
    B["sac_rate"] = B["sac_n"].fillna(0).where(has_fix) / B["minutes"]
    B["NS_switch_rate"] = B["NS_switches"] / B["minutes"]
    return B

## Build and save the feature table

Blocks with no fixation, saccade or pupil data are dropped (sensor or export failures).

In [4]:
FEATURES = ["North_dwell_frac", "South_dwell_frac", "North_fix_rate", "South_fix_rate",
            "North_fix_dur_mean", "South_fix_dur_mean",
            "pupil_mean", "pupil_sd", "pupil_mean_wp", "blink_frac", "fix_rate", "fix_dur_mean",
            "sac_rate", "sac_amp_mean", "sac_peakvel_px_mean", "sac_dur_mean",
            "SGE", "GTE", "p_N_to_S", "p_S_to_N", "NS_switch_rate"]

blocks = block_features(task)
n_before = len(blocks)
blocks = blocks.dropna(subset=FEATURES).reset_index(drop=True)
print(f"blocks: {n_before} built, {len(blocks)} with complete eye data | Commanders: {blocks['ID'].nunique()}")
print("blocks per Commander:", blocks.groupby("ID").size().value_counts().sort_index().to_dict())

TRUST = ["MDMT_North_C", "MDMT_South_C"]          # Commander trust in each UAV, rated once after the block
assert task.groupby(["ID", "Reliability"])[TRUST].nunique().max().max() <= 1, "trust rating varies within a block"
out = blocks[["ID", "Reliability", "Condition"] + FEATURES + TRUST].rename(columns={"Condition": "Priming"})
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
out.to_csv(OUT_PATH, index=False)
print("saved", OUT_PATH, out.shape)

blocks: 153 built, 140 with complete eye data | Commanders: 48
blocks per Commander: {1: 1, 2: 2, 3: 45}
saved data/eye_features_by_block.csv (140, 26)


In [5]:
# Sanity check: feature means by reliability condition
out.groupby("Reliability")[FEATURES].mean().loc[["HH", "M", "LL"]].T.round(3)

Reliability,HH,M,LL
North_dwell_frac,0.617,0.630,0.429
South_dwell_frac,0.322,0.313,0.512
North_fix_rate,71.854,74.886,53.861
South_fix_rate,38.841,36.403,52.773
North_fix_dur_mean,361.939,355.640,324.148
South_fix_dur_mean,347.878,356.672,398.024
pupil_mean,3.663,3.723,3.761
pupil_sd,0.232,0.240,0.234
pupil_mean_wp,-0.056,0.006,0.048
blink_frac,0.069,0.064,0.069
